# Ingestão streaming com Apache Kafka

Depende da camada Silver gerada no notebook 01, usada aqui como catálogo de municípios e como destino
dos eventos.

O resultado consolidado do indicador é anual, mas o processo que o produz não é. Durante a janela de
aplicação da avaliação, milhares de escolas transmitem resultados ao longo de semanas. Nesse período,
saber no mesmo dia se a aplicação está avançando nos municípios prioritários muda a decisão; saber no
ano seguinte, não. Daí a divisão: batch para o histórico consolidado, streaming para o que tem valor
perecível.

```
  escolas ──►  avaliacoes.raw ──►  processador  ──┬──► avaliacoes.enriched ──┬──► janelas + alertas
  (produtor)                    (valida/enriquece)│                          └──► micro-batch → lake
                                                  └──► dlq (dead letter queue)
```

Equivalência na AWS: Kafka vira MSK, o processador vira Lambda, o micro-batch vira Kinesis Firehose e
os alertas viram SNS.

---
## Etapa 1 - Kafka e o plano B

No Colab sobe um broker Kafka real, como nas aulas. Fora do Colab, num Windows sem Java por exemplo,
a instalação falharia e o notebook pararia na segunda célula.

Para a execução não depender do ambiente existe o `BrokerLocal`: um broker em memória com tópicos,
partições, offsets, consumer groups e lag, com a mesma interface de `KafkaProducer` e
`KafkaConsumer`. O restante do notebook é idêntico nos dois casos; muda apenas quem cria o cliente.

In [ ]:
# ============================================================
# ETAPA 1.1 — INSTALAÇÃO E START DO BROKER KAFKA (Colab)
# ============================================================
import os
import sys
import subprocess
import time

IN_COLAB = "google.colab" in sys.modules
KAFKA_VERSION, SCALA_VERSION = "3.7.0", "2.13"
KAFKA_DIR = f"/opt/kafka_{SCALA_VERSION}-{KAFKA_VERSION}"
BOOTSTRAP = "localhost:9092"


def subir_kafka_colab():
    """Baixa e inicia ZooKeeper + Kafka. Retorna True se o broker respondeu."""
    if not os.path.exists(KAFKA_DIR):
        print("Baixando Kafka...")
        subprocess.run("apt-get install -y openjdk-17-jre-headless -qq > /dev/null", shell=True)
        subprocess.run(
            f"wget -q https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/"
            f"kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz -O /tmp/kafka.tgz", shell=True)
        subprocess.run("tar -xzf /tmp/kafka.tgz -C /opt", shell=True)

    binarios = f"{KAFKA_DIR}/bin"
    subprocess.Popen(
        f"{binarios}/zookeeper-server-start.sh {KAFKA_DIR}/config/zookeeper.properties",
        shell=True, stdout=open("/tmp/zookeeper.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(10)
    subprocess.Popen(
        f"{binarios}/kafka-server-start.sh {KAFKA_DIR}/config/server.properties",
        shell=True, stdout=open("/tmp/kafka.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(15)

    saida = subprocess.run(f"{binarios}/kafka-topics.sh --bootstrap-server {BOOTSTRAP} --list",
                           shell=True, capture_output=True, text=True)
    return saida.returncode == 0


KAFKA_DISPONIVEL = False
if IN_COLAB:
    !pip install kafka-python-ng --quiet
    try:
        KAFKA_DISPONIVEL = subir_kafka_colab()
    except Exception as exc:
        print(f"Falha ao subir o Kafka: {exc}")

print(f"Broker Kafka real disponível: {KAFKA_DISPONIVEL}")
print("Modo de execução:", "KAFKA (MSK-like)" if KAFKA_DISPONIVEL else "BROKER LOCAL (em memória)")

In [ ]:
# ============================================================
# ETAPA 1.2 — CONTEXTO DO PROJETO (mesma raiz e mesmo lake do Notebook 01)
# ============================================================
import json
import logging
import shutil
import warnings
from collections import defaultdict, deque
from datetime import datetime, timezone, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

REPO_NAME = "tech-challenge-fase2"
REPO_URL  = "https://github.com/eriicpa/tech-challenge-fase2.git"


def localizar_raiz_projeto():
    for candidato in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidato / "data" / "landing").is_dir():
            return candidato
    if IN_COLAB:
        destino = Path("/content") / REPO_NAME
        if not destino.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(destino)], check=True)
        return destino
    raise RuntimeError("Raiz do projeto não encontrada — abra o notebook dentro do repositório.")


PROJECT_ROOT = localizar_raiz_projeto()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

LAKE            = Path(os.environ.get("TC2_LAKE", "data/lake"))
SILVER          = LAKE / "silver"
BRONZE_STREAM   = LAKE / "bronze_streaming"
QUALITY         = LAKE / "quality"
MONITOR         = LAKE / "monitoring"
for caminho in (BRONZE_STREAM, QUALITY, MONITOR):
    caminho.mkdir(parents=True, exist_ok=True)

PONTO_CORTE_SAEB = 743
DIM_REDE = {0: "Total", 1: "Federal", 2: "Estadual", 3: "Municipal", 4: "Privada", 5: "Pública"}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-8s | %(message)s",
                    datefmt="%Y-%m-%dT%H:%M:%SZ", force=True)
log = logging.getLogger("pipeline.streaming")

if not (SILVER / "dim_municipio").exists():
    raise RuntimeError("Camada Silver não encontrada. Execute o Notebook 01 antes deste.")

from src.utils.lakeio import ler_parquet

dim_municipio = ler_parquet(SILVER / "dim_municipio")
fato_indicador_municipio = ler_parquet(SILVER / "fato_indicador_municipio")

log.info("Silver carregada | municípios=%d | linhas de indicador=%d",
         len(dim_municipio), len(fato_indicador_municipio))
print(f"Raiz do projeto: {PROJECT_ROOT}")

In [ ]:
%%writefile src/streaming/broker_local.py
"""Um "Kafka de mentira" que roda em memoria.

Fiz isso pra conseguir rodar o notebook em qualquer maquina, porque instalar
Kafka de verdade so funciona no Colab/Linux. Tem so o que a pipeline usa:
topicos, particoes por hash da chave, offsets, consumer group e lag.

Os metodos tem os mesmos nomes do kafka-python (send, flush, commit...), entao
o resto do notebook nao precisa saber qual dos dois esta rodando.
"""
import json
import zlib
from collections import defaultdict
from datetime import datetime, timezone


class Mensagem:
    def __init__(self, topic, partition, offset, key, value, timestamp):
        self.topic = topic
        self.partition = partition
        self.offset = offset
        self.key = key
        self.value = value
        self.timestamp = timestamp


class MetadadosEnvio:
    def __init__(self, topic, partition, offset):
        self.topic = topic
        self.partition = partition
        self.offset = offset


class FuturoEnvio:
    """Imita o Future que o KafkaProducer.send() devolve."""

    def __init__(self, metadados):
        self._metadados = metadados

    def get(self, timeout=None):
        return self._metadados


class ProdutorLocal:

    def __init__(self, broker):
        self.broker = broker
        self.enviados = 0

    def send(self, topic, value=None, key=None):
        # serializa igual o Kafka faria, pra nao mascarar erro de serializacao
        valor = json.dumps(value).encode("utf-8")
        chave = str(key).encode("utf-8") if key is not None else None
        particao, offset = self.broker.produzir(topic, chave, valor)
        self.enviados += 1
        return FuturoEnvio(MetadadosEnvio(topic, particao, offset))

    def flush(self, timeout=None):
        return None


class ConsumidorLocal:

    def __init__(self, broker, topicos, group_id, auto_offset_reset="earliest",
                 consumer_timeout_ms=5000):
        self.broker = broker
        self.topicos = list(topicos)
        self.group_id = group_id

        # onde comecar a ler: do offset commitado, ou do inicio/fim se o grupo e novo
        self.posicoes = {}
        for topico in self.topicos:
            for particao in range(self.broker.numero_particoes(topico)):
                commitado = self.broker.offset_commitado(group_id, topico, particao)
                if commitado is not None:
                    inicio = commitado
                elif auto_offset_reset == "earliest":
                    inicio = 0
                else:
                    inicio = self.broker.offset_final(topico, particao)
                self.posicoes[(topico, particao)] = inicio

    def __iter__(self):
        for (topico, particao), posicao in sorted(self.posicoes.items()):
            for bruto in self.broker.ler(topico, particao, posicao):
                self.posicoes[(topico, particao)] = bruto["offset"] + 1
                yield Mensagem(
                    topic=topico,
                    partition=particao,
                    offset=bruto["offset"],
                    key=bruto["key"].decode("utf-8") if bruto["key"] else None,
                    value=json.loads(bruto["value"].decode("utf-8")),
                    timestamp=bruto["timestamp"],
                )

    def commit(self):
        for (topico, particao), posicao in self.posicoes.items():
            self.broker.commitar(self.group_id, topico, particao, posicao)

    def close(self):
        return None


class BrokerLocal:

    def __init__(self):
        self.topicos = {}
        self.commits = defaultdict(dict)

    def criar_topico(self, nome, particoes=1, replicacao=1):
        if nome not in self.topicos:
            self.topicos[nome] = {"particoes": particoes, "replicacao": replicacao,
                                  "dados": [[] for _ in range(particoes)]}
        return self.topicos[nome]

    def listar_topicos(self):
        return sorted(self.topicos)

    def numero_particoes(self, topico):
        return self.topicos[topico]["particoes"]

    def descrever(self, topico):
        info = self.topicos[topico]
        return [{"topico": topico, "particao": i, "mensagens": len(fila),
                 "ultimo_offset": len(fila) - 1 if fila else -1,
                 "replicacao": info["replicacao"]}
                for i, fila in enumerate(info["dados"])]

    def produzir(self, topico, chave, valor):
        if topico not in self.topicos:
            self.criar_topico(topico)
        # mesma chave sempre cai na mesma particao (e o que garante a ordem)
        total = self.numero_particoes(topico)
        particao = zlib.crc32(chave) % total if chave else 0

        fila = self.topicos[topico]["dados"][particao]
        offset = len(fila)
        fila.append({"offset": offset, "key": chave, "value": valor,
                     "timestamp": datetime.now(timezone.utc).timestamp()})
        return particao, offset

    def ler(self, topico, particao, desde_offset):
        return list(self.topicos[topico]["dados"][particao][desde_offset:])

    def offset_final(self, topico, particao):
        return len(self.topicos[topico]["dados"][particao])

    def commitar(self, grupo, topico, particao, offset):
        self.commits[grupo][(topico, particao)] = offset

    def offset_commitado(self, grupo, topico, particao):
        return self.commits.get(grupo, {}).get((topico, particao))

    def lag(self, grupo, topico):
        linhas = []
        for particao in range(self.numero_particoes(topico)):
            final = self.offset_final(topico, particao)
            atual = self.offset_commitado(grupo, topico, particao) or 0
            linhas.append({"topico": topico, "particao": particao, "grupo": grupo,
                           "offset_final": final, "offset_consumidor": atual,
                           "lag": final - atual})
        return linhas

    def produtor(self, **kwargs):
        return ProdutorLocal(self)

    def consumidor(self, topicos, group_id, **kwargs):
        return ConsumidorLocal(self, topicos, group_id, **kwargs)

In [ ]:
# ============================================================
# ETAPA 1.3 — TÓPICOS E CLIENTES (Kafka real ou broker local)
# ============================================================
from src.streaming.broker_local import BrokerLocal

TOPICO_RAW      = "alfabetizacao.avaliacoes.raw"
TOPICO_ENRICHED = "alfabetizacao.avaliacoes.enriched"
TOPICO_ALERTAS  = "alfabetizacao.alertas"
TOPICO_DLQ      = "alfabetizacao.dlq"

# O município é a chave da mensagem. Assim os eventos de um mesmo município caem
# sempre na mesma partição e a ordem é mantida.
TOPICOS = {
    TOPICO_RAW:      {"particoes": 4, "retencao_dias": 7},
    TOPICO_ENRICHED: {"particoes": 4, "retencao_dias": 7},
    TOPICO_ALERTAS:  {"particoes": 1, "retencao_dias": 30},
    TOPICO_DLQ:      {"particoes": 1, "retencao_dias": 30},
}

broker_local = None

if KAFKA_DISPONIVEL:
    from kafka import KafkaProducer, KafkaConsumer, KafkaAdminClient
    from kafka.admin import NewTopic
    from kafka.errors import TopicAlreadyExistsError

    admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
    for nome, config in TOPICOS.items():
        try:
            admin.create_topics([NewTopic(
                name=nome, num_partitions=config["particoes"], replication_factor=1,
                topic_configs={"retention.ms": str(config["retencao_dias"] * 86_400_000)})])
        except TopicAlreadyExistsError:
            pass

    def novo_produtor():
        return KafkaProducer(
            bootstrap_servers=BOOTSTRAP,
            value_serializer=lambda v: json.dumps(v).encode("utf-8"),
            key_serializer=lambda k: str(k).encode("utf-8") if k is not None else None,
            acks="all",            # durabilidade: confirma só quando as réplicas gravaram
            retries=3,
            linger_ms=20,          # micro-batching no produtor: menos requisições, mais throughput
            compression_type="gzip",
        )

    def novo_consumidor(topicos, grupo, desde_inicio=True, timeout_ms=8000):
        return KafkaConsumer(
            *topicos, bootstrap_servers=BOOTSTRAP, group_id=grupo,
            auto_offset_reset="earliest" if desde_inicio else "latest",
            enable_auto_commit=False,      # commit manual = at-least-once controlado
            value_deserializer=lambda b: json.loads(b.decode("utf-8")),
            key_deserializer=lambda b: b.decode("utf-8") if b else None,
            consumer_timeout_ms=timeout_ms,
        )

    def medir_lag(grupo, topico):
        from kafka import TopicPartition
        consumidor = KafkaConsumer(bootstrap_servers=BOOTSTRAP, group_id=grupo)
        particoes = [TopicPartition(topico, p) for p in consumidor.partitions_for_topic(topico)]
        finais = consumidor.end_offsets(particoes)
        linhas = []
        for tp in particoes:
            commitado = consumidor.committed(tp) or 0
            linhas.append({"topico": topico, "particao": tp.partition, "grupo": grupo,
                           "offset_final": finais[tp], "offset_consumidor": commitado,
                           "lag": finais[tp] - commitado})
        consumidor.close()
        return linhas

    def descrever_topicos():
        return pd.DataFrame([{"topico": nome, "particoes": cfg["particoes"],
                              "retencao_dias": cfg["retencao_dias"]}
                             for nome, cfg in TOPICOS.items()])

else:
    broker_local = BrokerLocal()
    for nome, config in TOPICOS.items():
        broker_local.criar_topico(nome, particoes=config["particoes"])

    def novo_produtor():
        return broker_local.produtor()

    def novo_consumidor(topicos, grupo, desde_inicio=True, timeout_ms=8000):
        return broker_local.consumidor(
            topicos, grupo,
            auto_offset_reset="earliest" if desde_inicio else "latest",
            consumer_timeout_ms=timeout_ms)

    def medir_lag(grupo, topico):
        return broker_local.lag(grupo, topico)

    def descrever_topicos():
        linhas = []
        for nome in broker_local.listar_topicos():
            linhas.extend(broker_local.descrever(nome))
        return pd.DataFrame(linhas)


print("Tópicos criados:\n")
display(descrever_topicos())

---
## Etapa 2 - Produtor de eventos

Os eventos são registros reais de aluno lidos da Silver. O que está simulado é o transporte, já que
na prática eles chegariam das escolas aos poucos.

Três decisões:

1. A chave da mensagem é o `id_municipio`. No Kafka a chave define a partição, então os eventos de um
   município caem sempre na mesma partição, o que garante ordem por município sem perder paralelismo
   entre municípios.
2. O evento carrega `ts_evento`, a hora do fato, além da hora de processamento. É o que permite medir
   latência real.
3. Dois problemas são injetados de propósito: 5% de eventos defeituosos, com campo faltante,
   proficiência impossível, município inexistente ou timestamp inválido, e uma queda de desempenho em
   5 municípios. O primeiro exercita a DLQ e o segundo, as regras de alerta.

Contrato do evento:

```json
{"id_evento": "uuid", "tipo_evento": "AVALIACAO_ALUNO", "ts_evento": "2026-08-27T14:32:10Z",
 "id_municipio": 3550308, "id_escola": "3550308007", "rede": 3, "serie": 2,
 "id_aluno": "AL2024000123", "proficiencia_portugues": 761.4, "peso_aluno": 1.09,
 "origem": "sistema_aplicacao_prova"}
```

In [ ]:
# ============================================================
# ETAPA 2.1 — PRODUTOR: REPRODUZINDO AVALIAÇÕES REAIS COMO EVENTOS
# ============================================================
import uuid

TOTAL_EVENTOS = 1_200
PCT_DEFEITUOSOS = 0.05
MUNICIPIOS_SIMULADOS = 40
MUNICIPIOS_EM_QUEDA = 5        # cenário injetado, para exercitar as regras de alerta
QUEDA_SIMULADA_PONTOS = 70

rng = np.random.default_rng(7)

# Os eventos são registros reais de aluno lidos da Silver. O que é simulado aqui
# é só o transporte: na prática eles chegariam das escolas aos poucos.
fato_aluno = ler_parquet(SILVER / "fato_aluno")
ano_recente = int(fato_aluno["ano"].max())

avaliados = fato_aluno[(fato_aluno["ano"] == ano_recente)
                       & (fato_aluno["rede"] == 3)
                       & fato_aluno["proficiencia_portugues"].notna()]

municipios_sorteados = (avaliados["id_municipio"].drop_duplicates()
                        .sample(n=MUNICIPIOS_SIMULADOS, random_state=7).tolist())
amostra = (avaliados[avaliados["id_municipio"].isin(municipios_sorteados)]
           .sample(n=min(TOTAL_EVENTOS, len(avaliados)), random_state=7)
           .reset_index(drop=True))

municipios_em_queda = set(rng.choice(municipios_sorteados, MUNICIPIOS_EM_QUEDA, replace=False))

log.info("Eventos a publicar: %d | municípios: %d | com queda injetada: %d",
         len(amostra), len(municipios_sorteados), len(municipios_em_queda))


def gerar_evento(registro, defeituoso=False):
    """Monta o evento a partir de um registro real de aluno."""
    proficiencia = float(registro["proficiencia_portugues"])
    if registro["id_municipio"] in municipios_em_queda:
        proficiencia = round(proficiencia - QUEDA_SIMULADA_PONTOS, 2)

    evento = {
        "id_evento": str(uuid.uuid4()),
        "tipo_evento": "AVALIACAO_ALUNO",
        "ts_evento": datetime.now(timezone.utc).isoformat(timespec="milliseconds"),
        "id_municipio": int(registro["id_municipio"]),
        "id_escola": str(registro["id_escola"]),
        "rede": int(registro["rede"]),
        "serie": int(registro["serie"]) if pd.notna(registro["serie"]) else 2,
        "id_aluno": str(registro["id_aluno"]),
        "proficiencia_portugues": proficiencia,
        "peso_aluno": float(registro.get("peso_aluno", 1.0)),
        "origem": "sistema_aplicacao_prova",
    }

    if defeituoso:
        problema = rng.choice(["campo_faltante", "fora_de_faixa",
                               "municipio_inexistente", "timestamp_invalido"])
        if problema == "campo_faltante":
            evento.pop("proficiencia_portugues")
        elif problema == "fora_de_faixa":
            evento["proficiencia_portugues"] = 2500.0
        elif problema == "municipio_inexistente":
            evento["id_municipio"] = 9_999_999
        else:
            evento["ts_evento"] = "ontem de manhã"
        evento["_defeito_injetado"] = problema
    return evento


produtor = novo_produtor()
inicio = time.perf_counter()
particoes_usadas = defaultdict(int)
defeituosos_enviados = 0

for _, registro in amostra.iterrows():
    defeituoso = rng.random() < PCT_DEFEITUOSOS
    defeituosos_enviados += int(defeituoso)
    evento = gerar_evento(registro, defeituoso)

    futuro = produtor.send(TOPICO_RAW, key=evento["id_municipio"], value=evento)
    particoes_usadas[futuro.get(timeout=10).partition] += 1

produtor.flush()
duracao = time.perf_counter() - inicio
TOTAL_EVENTOS = len(amostra)

print(f"Eventos publicados      : {TOTAL_EVENTOS:,}".replace(",", "."))
print(f"Base                    : registros reais de aluno (Silver, ano {ano_recente})")
print(f"Defeituosos (proposital): {defeituosos_enviados} "
      f"({defeituosos_enviados / TOTAL_EVENTOS:.1%})")
print(f"Municípios com queda injetada: {len(municipios_em_queda)}")
print(f"Tempo                   : {duracao:.2f}s | "
      f"{TOTAL_EVENTOS / duracao:,.0f} msg/s".replace(",", "."))
print("\nDistribuição entre partições (efeito da chave = id_municipio):")
display(pd.DataFrame(sorted(particoes_usadas.items()), columns=["particao", "mensagens"]))

---
## Etapa 3 - Validação, DLQ e enriquecimento

O processador valida cada evento contra o contrato, envia o que falhou para a DLQ com o motivo da
rejeição, e enriquece o que passou com a dimensão territorial e a regra dos 743 pontos.

A DLQ existe porque dado educacional errado não é apenas um bug: é município recebendo ou deixando de
receber recurso com base em número errado. Com ela o evento pode ser corrigido e reprocessado,
mantendo o registro de quem foi rejeitado e por quê.

O commit de offset é manual, feito só após processar com sucesso. É a semântica at-least-once: em
caso de falha a mensagem é reprocessada em vez de perdida, e a idempotência fica por conta do
`id_evento` na gravação no lake.

In [ ]:
# ============================================================
# ETAPA 3.1 — VALIDAÇÃO, DLQ E ENRIQUECIMENTO
# ============================================================
CAMPOS_OBRIGATORIOS = ["id_evento", "tipo_evento", "ts_evento", "id_municipio",
                       "rede", "proficiencia_portugues"]

# Os municípios ficam num dict em memória, para não consultar a cada evento.
CATALOGO_MUNICIPIOS = (dim_municipio.set_index("id_municipio")
                       [["nome_municipio", "sigla_uf", "nome_regiao"]].to_dict("index"))


def validar_evento(evento):
    """Devolve (valido, motivo). O motivo vai junto pra DLQ."""
    faltantes = [c for c in CAMPOS_OBRIGATORIOS if c not in evento or evento[c] is None]
    if faltantes:
        return False, f"campos_obrigatorios_ausentes:{','.join(faltantes)}"

    try:
        proficiencia = float(evento["proficiencia_portugues"])
    except (TypeError, ValueError):
        return False, "proficiencia_nao_numerica"
    if not 200 <= proficiencia <= 1000:
        return False, f"proficiencia_fora_de_faixa:{proficiencia}"

    if evento["id_municipio"] not in CATALOGO_MUNICIPIOS:
        return False, f"municipio_inexistente:{evento['id_municipio']}"

    try:
        datetime.fromisoformat(str(evento["ts_evento"]).replace("Z", "+00:00"))
    except ValueError:
        return False, "timestamp_invalido"

    return True, None


def enriquecer_evento(evento):
    """Marca se está alfabetizado e junta município/UF/região."""
    territorio = CATALOGO_MUNICIPIOS[evento["id_municipio"]]
    ts_evento = datetime.fromisoformat(str(evento["ts_evento"]).replace("Z", "+00:00"))
    agora = datetime.now(timezone.utc)

    return {
        **evento,
        "nome_municipio": territorio["nome_municipio"],
        "sigla_uf": territorio["sigla_uf"],
        "nome_regiao": territorio["nome_regiao"],
        "rede_nome": DIM_REDE.get(evento["rede"], "Desconhecida"),
        "alfabetizado": float(evento["proficiencia_portugues"]) >= PONTO_CORTE_SAEB,
        "ts_processamento": agora.isoformat(timespec="milliseconds"),
        "latencia_ms": round((agora - ts_evento).total_seconds() * 1000, 1),
    }


# ─── Loop do processador ───
consumidor_raw = novo_consumidor([TOPICO_RAW], grupo="processador-avaliacoes")
produtor_saida = novo_produtor()

processados, validos, rejeitados = 0, 0, 0
motivos_rejeicao = defaultdict(int)
latencias = []
inicio = time.perf_counter()

for mensagem in consumidor_raw:
    processados += 1
    evento = mensagem.value
    ok, motivo = validar_evento(evento)

    if not ok:
        rejeitados += 1
        motivos_rejeicao[motivo.split(":")[0]] += 1
        produtor_saida.send(TOPICO_DLQ, key=evento.get("id_municipio"), value={
            "evento_original": evento,
            "motivo_rejeicao": motivo,
            "topico_origem": mensagem.topic,
            "particao": mensagem.partition,
            "offset": mensagem.offset,
            "ts_rejeicao": datetime.now(timezone.utc).isoformat(timespec="milliseconds"),
        })
        continue

    enriquecido = enriquecer_evento(evento)
    latencias.append(enriquecido["latencia_ms"])
    produtor_saida.send(TOPICO_ENRICHED, key=enriquecido["id_municipio"], value=enriquecido)
    validos += 1

produtor_saida.flush()
consumidor_raw.commit()          # at-least-once: commit só após processar com sucesso
consumidor_raw.close()
duracao = time.perf_counter() - inicio

print(f"Processados : {processados:,}".replace(",", "."))
print(f"Válidos     : {validos:,} ({validos/max(processados,1):.1%})".replace(",", "."))
print(f"Rejeitados  : {rejeitados} -> {TOPICO_DLQ}")
print(f"Tempo       : {duracao:.2f}s  |  {processados/max(duracao,1e-9):,.0f} msg/s"
      .replace(",", "."))
print("\nMotivos de rejeição (o que a DLQ capturou):")
display(pd.DataFrame(sorted(motivos_rejeicao.items(), key=lambda x: -x[1]),
                     columns=["motivo", "eventos"]))

---
## Etapa 4 - Janelas, estado e alertas

Duas agregações. A janela temporal agrupa eventos em blocos fixos de tempo e mede throughput e
latência. A agregação com estado acumula por município e alimenta as regras de alerta.

Regras: `ABAIXO_DA_META` quando a taxa acumulada fica abaixo da meta com no mínimo 20 avaliações,
`QUEDA_DE_DESEMPENHO` quando fica 10 p.p. ou mais abaixo do indicador oficial do último ciclo, e
`LATENCIA_ACIMA_DO_SLA`.

A regra de queda compara o que chega agora com o indicador consolidado da Silver, produzido no
notebook 01. É o ponto em que batch e streaming se encontram; nenhum dos dois lados detectaria isso
sozinho.

Ressalva estatística: com cerca de 30 avaliações por município a taxa observada tem margem de erro
grande e a regra gera falsos positivos. Em produção o gatilho seria por intervalo de confiança. O
limiar foi mantido baixo para o mecanismo ficar visível na execução.

In [ ]:
# ============================================================
# ETAPA 4.1 — JANELAS TEMPORAIS + ESTADO POR MUNICÍPIO + ALERTAS
# ============================================================
JANELA_SEGUNDOS   = 1
MINIMO_AVALIACOES = 20      # abaixo disso, a taxa é ruído estatístico
QUEDA_ALERTA_PP   = 10.0    # p.p. abaixo do indicador oficial
SLA_LATENCIA_MS   = 2_000

# Linha de base vinda do batch (Silver), para comparar com o que chega no stream
baseline_oficial = (fato_indicador_municipio[
    (fato_indicador_municipio["ano"] == ano_recente) & (fato_indicador_municipio["rede"] == 3)]
    .set_index("id_municipio")["taxa_alfabetizacao"].to_dict())

consumidor_enriched = novo_consumidor([TOPICO_ENRICHED], grupo="agregador-janelas")
produtor_alertas = novo_produtor()

# Os eventos carregam peso_aluno, então a taxa do stream é calculada da mesma
# forma que o indicador oficial: média ponderada, não simples.
janelas = defaultdict(lambda: {"eventos": 0, "peso": 0.0, "peso_alfabetizados": 0.0,
                               "latencias": []})
estado_municipio = defaultdict(lambda: {"avaliados": 0, "peso": 0.0, "peso_alfabetizados": 0.0})
eventos_lidos = 0

for mensagem in consumidor_enriched:
    evento = mensagem.value
    eventos_lidos += 1

    # ── (a) Janela temporal pelo tempo do evento, não do processamento ──
    ts = datetime.fromisoformat(evento["ts_evento"].replace("Z", "+00:00"))
    inicio_janela = ts - timedelta(seconds=ts.second % JANELA_SEGUNDOS,
                                   microseconds=ts.microsecond)
    chave_janela = (inicio_janela.isoformat(timespec="seconds"), evento["sigla_uf"])
    peso = float(evento.get("peso_aluno", 1.0))
    janela = janelas[chave_janela]
    janela["eventos"] += 1
    janela["peso"] += peso
    janela["peso_alfabetizados"] += peso * int(evento["alfabetizado"])
    janela["latencias"].append(evento["latencia_ms"])

    # ── (b) Estado acumulado por município ──
    estado = estado_municipio[evento["id_municipio"]]
    estado["avaliados"] += 1
    estado["peso"] += peso
    estado["peso_alfabetizados"] += peso * int(evento["alfabetizado"])
    estado["nome_municipio"] = evento["nome_municipio"]
    estado["sigla_uf"] = evento["sigla_uf"]

consumidor_enriched.commit()
consumidor_enriched.close()

# ── Materialização das janelas ──
df_janelas = pd.DataFrame([
    {"janela_inicio": chave[0], "sigla_uf": chave[1], "eventos": dados["eventos"],
     "taxa_alfabetizacao_pct": round(dados["peso_alfabetizados"] / dados["peso"] * 100, 2),
     "latencia_p50_ms": round(float(np.percentile(dados["latencias"], 50)), 1),
     "latencia_p95_ms": round(float(np.percentile(dados["latencias"], 95)), 1)}
    for chave, dados in janelas.items()
]).sort_values(["janela_inicio", "eventos"], ascending=[True, False]).reset_index(drop=True)

# ── Avaliação das regras de alerta ──
alertas = []
for id_municipio, estado in estado_municipio.items():
    if estado["avaliados"] < MINIMO_AVALIACOES:
        continue
    taxa_stream = estado["peso_alfabetizados"] / estado["peso"] * 100
    taxa_oficial = baseline_oficial.get(id_municipio)

    if taxa_oficial is not None and taxa_stream < taxa_oficial - QUEDA_ALERTA_PP:
        alertas.append({
            "tipo": "QUEDA_DE_DESEMPENHO", "severidade": "ALTA", "id_municipio": id_municipio,
            "municipio": estado["nome_municipio"], "sigla_uf": estado["sigla_uf"],
            "taxa_stream_pct": round(taxa_stream, 2), "taxa_oficial_pct": round(taxa_oficial, 2),
            "delta_pp": round(taxa_stream - taxa_oficial, 2), "avaliados": estado["avaliados"],
        })
    elif taxa_stream < 50:
        alertas.append({
            "tipo": "ABAIXO_DA_META", "severidade": "MEDIA", "id_municipio": id_municipio,
            "municipio": estado["nome_municipio"], "sigla_uf": estado["sigla_uf"],
            "taxa_stream_pct": round(taxa_stream, 2),
            "taxa_oficial_pct": round(taxa_oficial, 2) if taxa_oficial else None,
            "delta_pp": None, "avaliados": estado["avaliados"],
        })

janelas_lentas = df_janelas[df_janelas["latencia_p95_ms"] > SLA_LATENCIA_MS]
for _, linha in janelas_lentas.iterrows():
    alertas.append({"tipo": "LATENCIA_ACIMA_DO_SLA", "severidade": "ALTA",
                    "id_municipio": None, "municipio": None, "sigla_uf": linha["sigla_uf"],
                    "taxa_stream_pct": None, "taxa_oficial_pct": None,
                    "delta_pp": linha["latencia_p95_ms"], "avaliados": int(linha["eventos"])})

for alerta in alertas:
    produtor_alertas.send(TOPICO_ALERTAS, key=alerta.get("sigla_uf"),
                          value={**alerta, "ts_alerta": datetime.now(timezone.utc).isoformat()})
produtor_alertas.flush()

print(f"Eventos agregados : {eventos_lidos:,}".replace(",", "."))
print(f"Janelas de {JANELA_SEGUNDOS}s     : {len(df_janelas)}")
print(f"Municípios com estado acumulado: {len(estado_municipio)}")
print(f"Alertas publicados em '{TOPICO_ALERTAS}': {len(alertas)}\n")
display(df_janelas.head(10))

In [ ]:
# ============================================================
# ETAPA 4.2 — CONSUMINDO O TÓPICO DE ALERTAS (quem seria notificado)
# ============================================================
# Na AWS, este consumidor é uma Lambda assinando o tópico e publicando no SNS:
# e-mail para a coordenação pedagógica, ticket para o time de dados.

consumidor_alertas = novo_consumidor([TOPICO_ALERTAS], grupo="notificador-alertas")
alertas_recebidos = [mensagem.value for mensagem in consumidor_alertas]
consumidor_alertas.commit()
consumidor_alertas.close()

if alertas_recebidos:
    df_alertas = pd.DataFrame(alertas_recebidos)
    print(f"ALERTAS ATIVOS: {len(df_alertas)}\n")
    display(df_alertas["tipo"].value_counts().rename_axis("tipo").reset_index(name="quantidade"))
    print("\nTop 10 alertas por severidade e distância da meta:")
    display(df_alertas.sort_values(["severidade", "delta_pp"])
            [["tipo", "severidade", "municipio", "sigla_uf", "taxa_stream_pct",
              "taxa_oficial_pct", "delta_pp", "avaliados"]].head(10))
else:
    print("Nenhum alerta disparado nesta execução.")

# ── Inspeção da DLQ: o que foi rejeitado e por quê ──
consumidor_dlq = novo_consumidor([TOPICO_DLQ], grupo="auditoria-dlq")
registros_dlq = [mensagem.value for mensagem in consumidor_dlq]
consumidor_dlq.commit()
consumidor_dlq.close()

print(f"\nDEAD LETTER QUEUE: {len(registros_dlq)} eventos aguardando correção\n")
if registros_dlq:
    df_dlq = pd.DataFrame([{
        "motivo": registro["motivo_rejeicao"].split(":")[0],
        "detalhe": registro["motivo_rejeicao"],
        "id_municipio": registro["evento_original"].get("id_municipio"),
        "defeito_injetado": registro["evento_original"].get("_defeito_injetado"),
    } for registro in registros_dlq])
    display(df_dlq["motivo"].value_counts().rename_axis("motivo").reset_index(name="eventos"))
    print("\nConferência: o motivo detectado bate com o defeito que injetamos?")
    display(pd.crosstab(df_dlq["defeito_injetado"], df_dlq["motivo"]))

---
## Etapa 5 - Do stream para o lake

Gravar evento a evento geraria milhares de arquivos minúsculos, o chamado small files problem, que
derruba a performance de leitura e infla o custo de requisições no S3.

A alternativa, e o que o Kinesis Firehose faz nativamente, é acumular em micro-batches por tempo ou
por tamanho, o que vier primeiro, e escrever um Parquet por lote.

O upsert na Silver usa `id_evento` como chave de deduplicação, o que torna o consumo at-least-once
seguro: reprocessar a mesma mensagem não duplica dado.

In [ ]:
# ============================================================
# ETAPA 5.1 — MICRO-BATCH PARA A BRONZE STREAMING
# ============================================================
TAMANHO_MICROBATCH = 500      # eventos
JANELA_MICROBATCH_S = 60      # ou 60 segundos, o que vier primeiro

destino_stream = BRONZE_STREAM / "avaliacoes"
if destino_stream.exists():
    shutil.rmtree(destino_stream)
destino_stream.mkdir(parents=True, exist_ok=True)

consumidor_lake = novo_consumidor([TOPICO_ENRICHED], grupo="sink-datalake")
buffer_eventos, lotes_gravados, eventos_gravados = [], 0, 0
abertura_lote = time.perf_counter()


def gravar_microbatch(eventos, numero_lote):
    """Grava um lote como Parquet particionado por data do evento."""
    df = pd.DataFrame(eventos)
    df["data_evento"] = pd.to_datetime(df["ts_evento"]).dt.strftime("%Y-%m-%d")
    df["_lote"] = numero_lote
    df["_ingestion_run_id"] = RUN_ID
    df.to_parquet(destino_stream, engine="pyarrow", compression="snappy",
                  partition_cols=["data_evento"], index=False)
    return len(df)


for mensagem in consumidor_lake:
    buffer_eventos.append(mensagem.value)
    estourou_tamanho = len(buffer_eventos) >= TAMANHO_MICROBATCH
    estourou_tempo = (time.perf_counter() - abertura_lote) >= JANELA_MICROBATCH_S
    if estourou_tamanho or estourou_tempo:
        eventos_gravados += gravar_microbatch(buffer_eventos, lotes_gravados)
        lotes_gravados += 1
        buffer_eventos, abertura_lote = [], time.perf_counter()
        consumidor_lake.commit()      # commit só depois de o dado estar durável no lake

if buffer_eventos:                    # flush do último lote parcial
    eventos_gravados += gravar_microbatch(buffer_eventos, lotes_gravados)
    lotes_gravados += 1
    consumidor_lake.commit()

consumidor_lake.close()

print(f"Micro-batches gravados : {lotes_gravados}")
print(f"Eventos persistidos    : {eventos_gravados:,}".replace(",", "."))
print(f"Média por lote         : {eventos_gravados / max(lotes_gravados, 1):.0f} eventos")
print(f"Destino                : {destino_stream}")

In [ ]:
# ============================================================
# ETAPA 5.2 — UPSERT NA SILVER (deduplicação por id_evento)
# ============================================================
from src.quality.data_quality import DataQualityCheck

stream_bronze = ler_parquet(destino_stream)
tabela_silver_stream = SILVER / "fato_avaliacao_stream"

# ── Transformação Silver: tipagem, seleção e derivações ──
silver_stream = stream_bronze[[
    "id_evento", "id_aluno", "ts_evento", "ts_processamento", "id_municipio", "nome_municipio",
    "sigla_uf", "nome_regiao", "id_escola", "rede", "rede_nome", "serie",
    "proficiencia_portugues", "peso_aluno", "alfabetizado", "latencia_ms", "data_evento",
]].copy()
silver_stream["id_municipio"] = silver_stream["id_municipio"].astype("int32")
silver_stream["proficiencia_portugues"] = silver_stream["proficiencia_portugues"].astype("float32")

# ── UPSERT: mescla com o que já existe e resolve por id_evento ──
if tabela_silver_stream.exists():
    existente = ler_parquet(tabela_silver_stream)
    antes = len(existente)
    combinado = pd.concat([existente, silver_stream], ignore_index=True)
else:
    antes = 0
    combinado = silver_stream

duplicados = int(combinado.duplicated(subset=["id_evento"]).sum())
combinado = combinado.drop_duplicates(subset=["id_evento"], keep="last").reset_index(drop=True)

if tabela_silver_stream.exists():
    shutil.rmtree(tabela_silver_stream)
tabela_silver_stream.mkdir(parents=True, exist_ok=True)
combinado.to_parquet(tabela_silver_stream, engine="pyarrow", compression="snappy",
                     partition_cols=["data_evento"], index=False)

print(f"Linhas antes do upsert : {antes:,}".replace(",", "."))
print(f"Linhas recebidas       : {len(silver_stream):,}".replace(",", "."))
print(f"Duplicatas resolvidas  : {duplicados}")
print(f"Linhas após o upsert   : {len(combinado):,}\n".replace(",", "."))

# ── Quality gate da camada streaming ──
dq_stream = (DataQualityCheck(combinado, "fato_avaliacao_stream", "silver", output_dir=QUALITY)
             .check_row_count(100)
             .check_unique(["id_evento"])
             .check_not_null(["id_evento", "id_municipio", "proficiencia_portugues", "sigla_uf"])
             .check_range("proficiencia_portugues", 200, 1000)
             .check_referential_integrity("id_municipio", dim_municipio, "id_municipio")
             .check_range("latencia_ms", 0, 60_000, severidade="WARNING"))
relatorio_stream = dq_stream.relatorio()
dq_stream.aprovar_ou_falhar()

# ── Idempotência: reaplicar o mesmo lote não pode alterar a tabela ──
reaplicado = pd.concat([combinado, silver_stream], ignore_index=True)
duplicatas_reaplicacao = int(reaplicado.duplicated(subset=["id_evento"]).sum())
reaplicado = reaplicado.drop_duplicates(subset=["id_evento"], keep="last")
print()
print(f"Reprocessamento do mesmo micro-batch: {duplicatas_reaplicacao} duplicatas descartadas | "
      f"linhas finais {len(reaplicado):,} (esperado: {len(combinado):,}, inalterado)"
      .replace(",", "."))

---
## Etapa 6 - Observabilidade

Em batch a pergunta é se o job rodou. Em streaming o job não termina, então a pergunta é se ele
acompanha o ritmo.

| Métrica | O que revela | Ação quando estoura |
|---|---|---|
| Consumer lag | quantas mensagens o consumidor está atrás | escalar consumidores, até o nº de partições |
| Latência p95 | atraso sentido por quem usa o dado | investigar gargalo do processador |
| Throughput | capacidade atual da pipeline | redimensionar partições |
| Taxa de DLQ | proporção de eventos inválidos | acionar o produtor do dado na origem |

O lag é a principal delas: é o único sinal que aparece antes do prejuízo, porque indica que a
pipeline processa mais devagar do que os dados chegam.

In [ ]:
# ============================================================
# ETAPA 6.1 — PAINEL DE OBSERVABILIDADE DO STREAMING
# ============================================================
lag_consolidado = pd.DataFrame(
    medir_lag("processador-avaliacoes", TOPICO_RAW)
    + medir_lag("agregador-janelas", TOPICO_ENRICHED)
    + medir_lag("sink-datalake", TOPICO_ENRICHED)
)

percentis = np.percentile(latencias, [50, 90, 95, 99]) if latencias else [0, 0, 0, 0]
taxa_dlq = rejeitados / max(processados, 1) * 100

metricas_streaming = {
    "run_id": RUN_ID,
    "modo": "kafka" if KAFKA_DISPONIVEL else "broker_local",
    "eventos_produzidos": TOTAL_EVENTOS,
    "eventos_processados": processados,
    "eventos_validos": validos,
    "eventos_dlq": rejeitados,
    "taxa_dlq_pct": round(taxa_dlq, 2),
    "throughput_msg_s": round(processados / max(duracao, 1e-9), 1),
    "latencia_p50_ms": round(float(percentis[0]), 1),
    "latencia_p90_ms": round(float(percentis[1]), 1),
    "latencia_p95_ms": round(float(percentis[2]), 1),
    "latencia_p99_ms": round(float(percentis[3]), 1),
    "lag_total": int(lag_consolidado["lag"].sum()),
    "janelas_processadas": len(df_janelas),
    "alertas_gerados": len(alertas),
    "microbatches_gravados": lotes_gravados,
    "gerado_em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}

destino_metricas = MONITOR / f"run_tc2-alfabetizacao-streaming_{RUN_ID}.json"
destino_metricas.write_text(json.dumps(metricas_streaming, indent=2, ensure_ascii=False),
                            encoding="utf-8")

print("=" * 74)
print("  PAINEL DE STREAMING — tc2-alfabetizacao-streaming")
print("=" * 74)
for chave, valor in metricas_streaming.items():
    print(f"  {chave:.<28} {valor}")
print("=" * 74)

# ── Semáforo operacional ──
sinais = [
    ("Consumer lag zerado", metricas_streaming["lag_total"] == 0,
     f"lag total = {metricas_streaming['lag_total']}"),
    ("Latência p95 dentro do SLA", metricas_streaming["latencia_p95_ms"] < SLA_LATENCIA_MS,
     f"p95 = {metricas_streaming['latencia_p95_ms']} ms (SLA {SLA_LATENCIA_MS} ms)"),
    ("Taxa de DLQ abaixo de 10%", taxa_dlq < 10,
     f"DLQ = {taxa_dlq:.2f}%"),
    ("Todos os eventos válidos persistidos", eventos_gravados == validos,
     f"{eventos_gravados} gravados / {validos} válidos"),
]
print("\nSEMÁFORO OPERACIONAL")
display(pd.DataFrame([{"verificação": nome, "status": "OK" if ok else "ATENÇÃO", "detalhe": detalhe}
                      for nome, ok, detalhe in sinais]))

print("\nCONSUMER LAG POR PARTIÇÃO")
display(lag_consolidado)

---
## Etapa 7 - Equivalência na AWS e custo

| Componente aqui | Serviço AWS |
|---|---|
| Broker Kafka | MSK Serverless, mesma API, sem gestão de cluster |
| Processador | Lambda com gatilho MSK |
| Micro-batch para o S3 | Kinesis Firehose, com buffer e conversão para Parquet |
| Tópico de alertas | SNS |
| DLQ | SQS DLQ |
| Lag e métricas | CloudWatch, com alarme quando o lag cresce |

Sobre custo: streaming cobra por tempo ligado e batch cobra por volume processado. Daí a pipeline ser
híbrida. O broker fica ativo só na janela de aplicação, algumas semanas por ano; o MSK é Serverless
em vez de cluster provisionado; e a retenção do tópico é de 7 dias, com o histórico definitivo no S3,
que é bem mais barato que armazenamento de broker.

---
## Artefatos gerados

```
data/lake/bronze_streaming/avaliacoes/   micro-batches em Parquet
data/lake/silver/fato_avaliacao_stream/  tabela deduplicada por id_evento
data/lake/quality/                       relatório de qualidade do stream
data/lake/monitoring/                    lag, latência, throughput e taxa de DLQ

src/streaming/broker_local.py            broker em memória compatível com kafka-python
```

O notebook 03 constrói a camada Gold, gera os jobs AWS Glue e os DDLs do Athena, e faz a análise de
FinOps.